# Banana Leaf Disease Classification Using Deep Learning
Step 0: Import libraries and set reproducible configuration settings.

In [1]:
import hashlib, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from PIL import Image, ImageFile
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score, log_loss

warnings.filterwarnings("ignore", category=UserWarning, module="PIL")
ImageFile.LOAD_TRUNCATED_IMAGES = True          # a few JPEGs in the dataset are slightly truncated
sns.set_theme(style="whitegrid")

SEED, IMG_SIZE, BATCH, NOISE_STD = 42, (224, 224), 32, 10.0
DATA_DIR    = Path("OriginalSet")                # dataset folder next to this notebook
CLASS_NAMES = ["cordana", "healthy", "pestalotiopsis", "sigatoka"]
CLASS_TO_ID = {c: i for i, c in enumerate(CLASS_NAMES)}
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)

def reset_seed(seed=SEED):
    keras.utils.set_random_seed(seed)                # Python + NumPy + TensorFlow seeds
reset_seed()
tf.config.experimental.enable_op_determinism()

assert DATA_DIR.is_dir(), "Folder 'OriginalSet' not found: put it next to this notebook."
print("TensorFlow", tf.__version__, "| GPU:", bool(tf.config.list_physical_devices("GPU")), "| dataset:", DATA_DIR.resolve())

Error: No connection selected.

## Task 1 : Load and explore the dataset

**Step 1.1:** Load every image path, label and pixels.

In [2]:
rows = [(str(p), p.name, c, CLASS_TO_ID[c], hashlib.md5(p.read_bytes()).hexdigest())
        for c in CLASS_NAMES for p in sorted((DATA_DIR / c).glob("*")) if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
df = pd.DataFrame(rows, columns=["filepath", "filename", "label", "label_id", "md5"])
images = [np.asarray(Image.open(p).convert("RGB")) for p in df.filepath]

print(f"Loaded {len(images)} images; folder names used as labels: {CLASS_TO_ID}")
print("Raw image shapes:", pd.Series([a.shape for a in images]).value_counts().to_dict())
df.head()

Error: No connection selected.

**Step 1.2:** Display labeled sample images from all classes.

In [3]:
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for r, c in enumerate(CLASS_NAMES):
    for ax, i in zip(axes[r], df[df.label == c].sample(4, random_state=SEED).index):
        ax.imshow(images[i]); ax.set_title(f"{c} | {df.filename[i]}", fontsize=9); ax.axis("off")
plt.suptitle("Four labeled samples per class", fontsize=14); plt.tight_layout(); plt.show()

Error: No connection selected.

**Step 1.3:** Count images per class and discuss balance.

In [4]:
counts = df.label.value_counts().reindex(CLASS_NAMES)
display(pd.DataFrame({"images": counts, "share_%": (100 * counts / len(df)).round(1)}))
sns.barplot(x=counts.index, y=counts.values, hue=counts.index, legend=False)
plt.title("Images per class"); plt.ylabel("images"); plt.show()

n_dup = int(df.md5.duplicated().sum())
print(f"DISCUSSION\n- Total {len(df)} images; largest/smallest class ratio = {counts.max() / counts.min():.1f}. "
      f"'{counts.idxmax()}' dominates ({100 * counts.max() / len(df):.0f} %), '{counts.idxmin()}' is the smallest.")
print("- The dataset is imbalanced: a model could look accurate by favouring the big class, "
      "so I use stratified splits, class weights and per-class metrics (recall, F1).")
print(f"- {n_dup} files are exact copies of another file. They are KEPT; in Step 2.3 all copies of one image "
      "are forced into the same split, so no image appears in more than one split.")

Error: No connection selected.

## Task 2 : Preprocess the images

**Step 2.1:** Resize every image to one consistent size.

In [5]:
X = np.stack([np.asarray(Image.fromarray(a).resize(IMG_SIZE, Image.Resampling.BILINEAR)) for a in images]).astype("uint8")
y = df.label_id.to_numpy()
print(f"All images resized to {IMG_SIZE} -> tensor {X.shape}, dtype {X.dtype}, labels {y.shape}")

Error: No connection selected.

**Step 2.2:** Scale pixel values the way MobileNetV2 expects.

In [6]:
sample = X[:8].astype("float32")
scaled = preprocess_input(sample.copy())
print(f"Before: {sample.min():.0f}..{sample.max():.0f}  ->  After preprocess_input: {scaled.min():.2f}..{scaled.max():.2f} "
      "(MobileNetV2 was pre-trained on inputs scaled to [-1, 1]; applied inside the data pipeline).")

Error: No connection selected.

**Step 2.3:** Split into training, validation and test sets.

In [7]:

sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
folds = [te for _, te in sgkf.split(df, y, groups=df.md5.factorize()[0])]
idx = {"train": np.concatenate(folds[2:]), "val": folds[1], "test": folds[0]}

(X_train, y_train), (X_val, y_val), (X_test, y_test) = [(X[idx[k]], y[idx[k]]) for k in ("train", "val", "test")]
test_names = df.filename.to_numpy()[idx["test"]]

for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    assert not (set(idx[a]) & set(idx[b])) and not (set(df.md5[idx[a]]) & set(df.md5[idx[b]])), "leakage!"
print("✔ No image (or identical copy) appears in more than one split.\n")

sizes = pd.DataFrame({k: pd.Series(y[v]).map(dict(enumerate(CLASS_NAMES))).value_counts() for k, v in idx.items()}).reindex(CLASS_NAMES)
sizes.loc["TOTAL"] = sizes.sum()
display(sizes)
print({k: f"{len(v)} ({100 * len(v) / len(df):.1f} %)" for k, v in idx.items()})

Error: No connection selected.

**Step 2.4:** Augment only training images with Gaussian noise.

In [8]:
augmenter = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"), layers.RandomRotation(0.15, fill_mode="reflect"),
    layers.RandomZoom(0.2, fill_mode="reflect"), layers.RandomTranslation(0.1, 0.1, fill_mode="reflect"),
    layers.RandomContrast(0.25), layers.RandomBrightness(0.2, value_range=(0, 255)),
])
cw = compute_class_weight("balanced", classes=np.arange(len(CLASS_NAMES)), y=y_train)   # from training labels only
cw_vec = tf.constant(cw, tf.float32)
def make_aug():
    # fresh layers with fixed seeds: every dataset gets the same random sequence on every run
    aug = keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical", seed=1), layers.RandomRotation(0.15, fill_mode="reflect", seed=2),
        layers.RandomZoom(0.2, fill_mode="reflect", seed=3), layers.RandomTranslation(0.1, 0.1, fill_mode="reflect", seed=4),
        layers.RandomContrast(0.25, seed=5), layers.RandomBrightness(0.2, value_range=(0, 255), seed=6),
    ])
    return aug, layers.GaussianNoise(NOISE_STD / 255.0, seed=7)       # same noise strength (10 on the 0-255 scale)

cw = compute_class_weight("balanced", classes=np.arange(len(CLASS_NAMES)), y=y_train)   # from training labels only
cw_vec = tf.constant(cw, tf.float32)

def make_ds(images, labels, training):
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    aug, noise = make_aug() if training else (None, None)
    if training:
        ds = ds.shuffle(len(images), seed=SEED)
    def prep(x, y_):
        x = tf.cast(x, tf.float32)
        if training:                                                    # augmentation + noise: TRAIN ONLY
            x = aug(x, training=True)
            x = tf.clip_by_value(noise(x / 255.0, training=True) * 255.0, 0.0, 255.0)
            return preprocess_input(x), y_, tf.gather(cw_vec, y_)       # (image, label, class weight)
        return preprocess_input(x), y_
    return ds.batch(BATCH).map(prep).prefetch(tf.data.AUTOTUNE)         # map without parallel calls = fixed order

train_ds, val_ds, test_ds = make_ds(X_train, y_train, True), make_ds(X_val, y_val, False), make_ds(X_test, y_test, False)

xb, yb, _ = next(iter(make_ds(X_train, y_train, True)))                 # preview uses its OWN dataset, so training is not affected
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
for ax, im, lab in zip(axes, xb.numpy(), yb.numpy()):
    ax.imshow((im + 1) / 2); ax.set_title(f"train aug: {CLASS_NAMES[lab]}", fontsize=9); ax.axis("off")
plt.show()
print("Class weights:", dict(zip(CLASS_NAMES, cw.round(2))), "| validation/test pipelines: no augmentation, no noise.")


Error: No connection selected.

## Task 3 : Build the model (transfer learning)

**Step 3.1:** Build MobileNetV2 transfer learning model with softmax.

In [9]:

reset_seed()
base = MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
base.trainable = False                                                  # stage 1: backbone frozen

inputs  = keras.Input(IMG_SIZE + (3,), name="leaf_image")
x = layers.GlobalAveragePooling2D()(base(inputs, training=False))
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4))(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(len(CLASS_NAMES), activation="softmax", name="class_probabilities")(x)
model = keras.Model(inputs, outputs, name="banana_leaf_mobilenetv2")
print("Input:", model.input_shape, "| Output:", model.output_shape)


Error: No connection selected.

**Step 3.2:** Display the architecture and explain each layer.

In [10]:
model.summary()
print('''EXPLANATION
- Input 224x224x3: an RGB leaf image. Output Dense(4, softmax): one probability per class (cordana, healthy, pestalotiopsis, sigatoka).
- MobileNetV2 (ImageNet weights): depth-wise separable convolutions extract edges, textures and lesion patterns; frozen first, fine-tuned later.
- GlobalAveragePooling2D: averages each feature map -> 1280 numbers, far fewer parameters than Flatten, less overfitting.
- Dense(256, ReLU) + L2 + two Dropout layers: small trainable classifier head that is regularised against overfitting.''')

Error: No connection selected.

## Task 4 : Train the model

**Step 4.1:** Configure Adam optimizer, loss, metrics and callbacks.

In [12]:
def compile_model(lr):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])

def make_callbacks(patience):
    return [keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True, verbose=1),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7, verbose=1)]

compile_model(1e-3)
print("Loss: sparse categorical cross-entropy | Optimizer: Adam | Metric: accuracy | EarlyStopping on val_loss (best weights restored)")
print("Plan: stage 1 up to 10 epochs (head only) + stage 2 up to 20 more epochs (fine-tuning) = max 30 epochs.")

Error: No connection selected.

**Step 4.2:** Train only the new classification head first.

In [13]:
h1 = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=make_callbacks(4))
n1 = len(h1.history["loss"])
print(f"Stage 1 finished after {n1} epoch(s).")

Error: No connection selected.

**Step 4.3:** Unfreeze top backbone layers and tune gently.

In [14]:
base.trainable = True
for layer in base.layers[:-120]:                                       # was [:-80]: unfreeze more layers
    layer.trainable = False
for layer in base.layers:                                              # BatchNorm stays frozen (stable fine-tuning)
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
compile_model(3e-5)                                                    # was 2e-5
h2 = model.fit(train_ds, validation_data=val_ds, epochs=30, initial_epoch=n1, callbacks=make_callbacks(6))

Error: No connection selected.

**Step 4.4:** Report epochs, explain early stopping, plot curves.

In [15]:
hist = pd.DataFrame({k: h1.history[k] + h2.history[k] for k in ["accuracy", "val_accuracy", "loss", "val_loss"]})
hist.index += 1; hist.index.name = "epoch"
display(hist.round(4))
total, best = len(hist), int(hist.val_loss.idxmin())
print(f"Epochs run: {total} (max 30) | best epoch (lowest val_loss): {best}")
print(f"EARLY STOPPING: validation loss did not improve after epoch {best}; training stopped at epoch {total} "
      f"and the weights of epoch {best} were restored." if total < 30 else "Training used the full 30-epoch budget.")
if total < 10: print("Training ended before 10 epochs - the full history above is shown as required.")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, m in zip(axes, ["accuracy", "loss"]):
    ax.plot(hist.index, hist[m], "o-", label=f"train {m}"); ax.plot(hist.index, hist["val_" + m], "s-", label=f"val {m}")
    ax.axvline(n1 + 0.5, color="gray", ls=":", label="fine-tuning starts"); ax.axvline(best, color="green", ls="--", label="best epoch")
    ax.set_title(f"Training vs validation {m}"); ax.set_xlabel("epoch"); ax.legend()
plt.tight_layout(); plt.show()

g = hist.accuracy[best] - hist.val_accuracy[best]
print(f"INTERPRETATION\n- At epoch {best}: train acc {hist.accuracy[best]:.3f}, val acc {hist.val_accuracy[best]:.3f} (gap {g:+.3f}); "
      f"train loss {hist.loss[best]:.3f}, val loss {hist.val_loss[best]:.3f}.")
print("- Training metrics are measured on augmented, noisy images with dropout on, so they are harder than validation images.")
print("- Small gap and a flat validation curve = good generalisation." if g < 0.08 else "- Large gap = signs of overfitting.")
print(f"- Fine-tuning (from epoch {n1 + 1}) changed best val loss from {hist.val_loss[:n1].min():.3f} to {hist.val_loss.min():.3f}.")

Error: No connection selected.

## Task 5 : Evaluate the model

**Step 5.1:** Evaluate unseen test data: loss and accuracy.

In [16]:
def predict_tta(a):
    ds = lambda v: tf.data.Dataset.from_tensor_slices(np.ascontiguousarray(v)).batch(BATCH).map(
        lambda t: preprocess_input(tf.cast(t, tf.float32)))
    views = (a, a[:, :, ::-1], a[:, ::-1], a[:, ::-1, ::-1])
    return np.mean([model.predict(ds(v), verbose=0) for v in views], axis=0)

data  = {"train": (X_train, y_train), "val": (X_val, y_val), "test": (X_test, y_test)}
prob  = {k: predict_tta(v[0]) for k, v in data.items()}
pred  = {k: p.argmax(1) for k, p in prob.items()}
y_prob, y_pred = prob["test"], pred["test"]

plain_loss, plain_acc = model.evaluate(test_ds, verbose=0)
test_loss, test_acc = log_loss(y_test, y_prob, labels=range(4)), accuracy_score(y_test, y_pred)
print(f"Plain model        -> test loss {plain_loss:.4f} | test accuracy {plain_acc:.4f}")
print(f"FINAL (with flips) -> TEST LOSS {test_loss:.4f} | TEST ACCURACY {test_acc:.4f} ({test_acc * 100:.2f} %)")
print(f"Macro F1: {f1_score(y_test, y_pred, average='macro'):.4f}")

Error: No connection selected.

**Step 5.2:** Plot confusion matrix with all class names.

In [17]:
cm = confusion_matrix(y_test, y_pred, labels=range(4))
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0], cbar=False)
sns.heatmap(cm / cm.sum(1, keepdims=True), annot=True, fmt=".2f", cmap="Greens", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
for ax, t in zip(axes, ["Confusion matrix (counts)", "Confusion matrix (row-normalised)"]):
    ax.set_title(t); ax.set_xlabel("Predicted class"); ax.set_ylabel("True class")
plt.tight_layout(); plt.show()

Error: No connection selected.

**Step 5.3:** Print precision, recall and F1 per class.

In [18]:
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

Error: No connection selected.

**Step 5.4:** Analyze errors and diagnose overfitting or underfitting.

In [19]:
wrong = pd.DataFrame({"file": test_names, "true": [CLASS_NAMES[i] for i in y_test], "pred": [CLASS_NAMES[i] for i in y_pred],
                      "confidence": y_prob.max(1).round(3)}).query("true != pred")
print(f"ERRORS: {len(wrong)} of {len(y_test)} test images misclassified.")
if len(wrong):
    print(wrong.groupby(["true", "pred"]).size().sort_values(ascending=False).to_string())
    print(f"Mean confidence of wrong predictions: {wrong.confidence.mean():.2f} (lower = the model is often unsure when it errs).")
rec = pd.Series(np.diag(cm) / cm.sum(1), index=CLASS_NAMES)
print(f"Weakest class by recall: {rec.idxmin()} ({rec.min():.2f}). Errors occur mostly between visually similar lesions.")

acc = {k: accuracy_score(data[k][1], pred[k]) for k in data}
print("\nAccuracy (clean data, same predictor):", {k: round(v, 4) for k, v in acc.items()})
if acc["train"] < 0.80 and acc["test"] < 0.80:
    verdict = "UNDERFITTING: accuracy is low even on training data."
elif acc["train"] - acc["test"] > 0.08:
    verdict = "OVERFITTING: training accuracy is much higher than test accuracy."
else:
    verdict = "GOOD FIT: high accuracy on all sets with a small train-test gap (no significant over/underfitting)."
print("CONCLUSION:", verdict)

Error: No connection selected.

## Task 6 : Make and save predictions

**Step 6.1:** Show test images: true versus predicted class.

In [20]:
show = np.random.default_rng(SEED).choice(len(X_test), 12, replace=False)
fig, axes = plt.subplots(3, 4, figsize=(13, 10))
for ax, i in zip(axes.ravel(), show):
    ax.imshow(X_test[i]); ax.axis("off")
    ax.set_title(f"True: {CLASS_NAMES[y_test[i]]}\nPred: {CLASS_NAMES[y_pred[i]]} ({y_prob[i].max():.0%})",
                 color="green" if y_test[i] == y_pred[i] else "red", fontsize=10)
plt.suptitle("Test predictions (green = correct, red = wrong)", fontsize=14); plt.tight_layout(); plt.show()

Error: No connection selected.

**Step 6.2:** Save the trained model and print name.

In [21]:
MODEL_PATH = OUT / "banana_leaf_mobilenetv2.keras"
model.save(MODEL_PATH)
same = np.allclose(keras.models.load_model(MODEL_PATH).predict(test_ds, verbose=0), model.predict(test_ds, verbose=0), atol=1e-5)
print(f"✔ Model saved as: {MODEL_PATH.name}  | reloaded model gives identical predictions: {same}")

Error: No connection selected.

**Step 6.3:** Save predictions CSV with filename, true, predicted.

In [22]:
pred_df = pd.DataFrame({"image_filename": test_names, "true_class": [CLASS_NAMES[i] for i in y_test],
                        "predicted_class": [CLASS_NAMES[i] for i in y_pred], "confidence": y_prob.max(1).round(4)})
pred_df.to_csv(OUT / "test_predictions.csv", index=False)
print(f"✔ Predictions saved as: test_predictions.csv ({len(pred_df)} test images) | accuracy {test_acc * 100:.2f} %")
pred_df.head(10)

Error: No connection selected.